# Customer Churn Prediction

In [1]:
import pandas as pd
import numpy as np

In [2]:
raw_telecom = pd.read_csv('Telco-Customer-Churn.csv')

In [3]:
raw_telecom.shape

(7043, 21)

## Creating a copy of a raw data

In [4]:
telecom = raw_telecom.copy()

In [5]:
telecom.shape

(7043, 21)

In [6]:
telecom.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [7]:
telecom.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


### Handling missing values and Standardising data

In [15]:
telecom['TotalCharges'] = telecom['TotalCharges'].replace(' ', np.nan)

In [16]:
telecom['TotalCharges'].isnull().sum()

np.int64(11)

In [17]:
telecom.dropna(inplace=True)

In [ ]:
telecom.shape

(7032, 21)

In [20]:
telecom.duplicated().sum()

np.int64(0)

In [23]:
for col in telecom.select_dtypes(include='object').columns:
    telecom[col] = telecom[col].str.strip()

In [27]:
for col in telecom.select_dtypes(include='object').columns:
    print(col)
    print(telecom[col].unique())
    print()

customerID
['7590-VHVEG' '5575-GNVDE' '3668-QPYBK' ... '4801-JZAZL' '8361-LTMKD'
 '3186-AJIEK']

gender
['Female' 'Male']

MultipleLines
['No phone service' 'No' 'Yes']

InternetService
['DSL' 'Fiber optic' 'No']

OnlineSecurity
['No' 'Yes' 'No internet service']

OnlineBackup
['Yes' 'No' 'No internet service']

DeviceProtection
['No' 'Yes' 'No internet service']

TechSupport
['No' 'Yes' 'No internet service']

StreamingTV
['No' 'Yes' 'No internet service']

StreamingMovies
['No' 'Yes' 'No internet service']

Contract
['Month-to-month' 'One year' 'Two year']

PaymentMethod
['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)']



In [25]:
telecom['TotalCharges'] = telecom['TotalCharges'].astype(float)

In [26]:
binary_cols = ['Partner', 'Dependents', 'PhoneService', 
               'PaperlessBilling', 'Churn']

for col in binary_cols:
    telecom[col] = telecom[col].map({
        'Yes': 1,
        'No': 0
    })

In [28]:
telecom.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7032 non-null   object 
 1   gender            7032 non-null   object 
 2   SeniorCitizen     7032 non-null   int64  
 3   Partner           7032 non-null   int64  
 4   Dependents        7032 non-null   int64  
 5   tenure            7032 non-null   int64  
 6   PhoneService      7032 non-null   int64  
 7   MultipleLines     7032 non-null   object 
 8   InternetService   7032 non-null   object 
 9   OnlineSecurity    7032 non-null   object 
 10  OnlineBackup      7032 non-null   object 
 11  DeviceProtection  7032 non-null   object 
 12  TechSupport       7032 non-null   object 
 13  StreamingTV       7032 non-null   object 
 14  StreamingMovies   7032 non-null   object 
 15  Contract          7032 non-null   object 
 16  PaperlessBilling  7032 non-null   int64  
 17  

In [36]:
telecom.shape

(7032, 21)

## Feature Engineering

### Tenure Groups

In [37]:
telecom['TenureGroup'] = pd.cut(
    telecom['tenure'],
    bins=[0,12,24,48,72],
    labels=['0-1 Year','1-2 Years','2-4 Years','4-6 Years']
)

In [64]:
telecom['TenureGroup'].value_counts()

TenureGroup
4-6 Years    2239
0-1 Year     2175
2-4 Years    1594
1-2 Years    1024
Name: count, dtype: int64

### Average Monthly Spend 

In [46]:
telecom['AvgMonthlySpend'] = (
    telecom['TotalCharges'] / (telecom['tenure'])
)

In [51]:
telecom['AvgMonthlySpend'].describe()

count    7032.000000
mean       64.799424
std        30.185891
min        13.775000
25%        36.179891
50%        70.373239
75%        90.179560
max       121.400000
Name: AvgMonthlySpend, dtype: float64

### Number of Services active

In [58]:
telecom['NumServices'] = (
    (telecom['PhoneService'] == 'Yes').astype(int) +
    (telecom['MultipleLines'] == 'Yes').astype(int) +
    (telecom['OnlineSecurity'] == 'Yes').astype(int) +
    (telecom['OnlineBackup'] == 'Yes').astype(int) +
    (telecom['DeviceProtection'] == 'Yes').astype(int) +
    (telecom['TechSupport'] == 'Yes').astype(int) +
    (telecom['StreamingTV'] == 'Yes').astype(int) +
    (telecom['StreamingMovies'] == 'Yes').astype(int)
)

In [66]:
telecom['NumServices'].value_counts()

NumServices
0    1663
1    1156
3     978
2     957
4     931
5     719
6     420
7     208
Name: count, dtype: int64

### Risky Contracts (Might churn)

In [67]:
telecom['HighRiskContract'] = (
    telecom['Contract'] == 'Month-to-month'
).astype(int)

In [69]:
telecom['HighRiskContract'].value_counts()

HighRiskContract
1    3875
0    3157
Name: count, dtype: int64

### Auto Payment (Won't churn)

In [71]:
telecom['AutoPayment'] = (
    telecom['PaymentMethod'].isin([
        'Bank transfer (automatic)',
        'Credit card (automatic)'
    ])
).astype(int)

In [74]:
telecom['AutoPayment'].value_counts()

AutoPayment
0    3969
1    3063
Name: count, dtype: int64

### Fiber users (Might churn)

In [ ]:
telecom['FiberOpticUser'] = (
    telecom['InternetService'] == 'Fiber optic'
).astype(int)

In [78]:
telecom['FiberOpticUser'].value_counts()

FiberOpticUser
0    3936
1    3096
Name: count, dtype: int64

### Customers with family (Won't turn)

In [87]:
telecom['FamilyCustomer'] = (
    (telecom['Partner'] == 1) |
    (telecom['Dependents'] == 1)
).astype(int)

In [89]:
telecom['FamilyCustomer'].value_counts()

FamilyCustomer
1    3752
0    3280
Name: count, dtype: int64

### Senior Citizen feature

In [5]:
telecom['SeniorHighCharges'] = (
    telecom['SeniorCitizen'] * telecom['MonthlyCharges']
)

### Long term Customers (Won't churn)

In [8]:
telecom['LongTermCustomer'] = (
    telecom['tenure'] >= 24
).astype(int)

In [9]:
telecom['LongTermCustomer'].value_counts()

LongTermCustomer
1    3927
0    3105
Name: count, dtype: int64

### Service Value

In [12]:
telecom['ServiceValueScore'] = (
    telecom['NumServices'] / (telecom['MonthlyCharges'])
)

In [15]:
telecom['ServiceValueScore'].describe()

count    7032.000000
mean        0.032401
std         0.023875
min         0.000000
25%         0.012618
50%         0.034173
75%         0.048745
max         0.096000
Name: ServiceValueScore, dtype: float64

In [16]:
telecom.to_csv('telecom_cleanedwithfeatures.csv', index=False)